# Deploying the UnderCover app to AWS Lambda

This notebook takes `server.py`, a small FastAPI app, and puts it on AWS Lambda behind a public Function URL. It builds the package, deploys it, calls it, shows what now exists in the account, then tears it all down and shows that nothing is left.

What it creates, all named `undercover` and tagged `project=undercover`:

- an IAM role, `undercover-app-role`, that the function runs as
- a Lambda function, `undercover-app`, with a Function URL
- a log group, `/aws/lambda/undercover-app`, where the function writes its logs

The app's claims routes need an API key, which this notebook keeps in `.env` and passes to the function.

Lambda's always-free allowance is a million requests a month, so this costs nothing.

## 1. Setup

boto3 reads the same credentials as the AWS CLI, so there are no keys in this notebook.

In [ ]:
import json
import os
import secrets
import shutil
import time
from pathlib import Path

import boto3
import requests
from dotenv import load_dotenv

PREFIX = "undercover"
FUNCTION_NAME = "undercover-app"
ROLE_NAME = "undercover-app-role"
LOG_GROUP = f"/aws/lambda/{FUNCTION_NAME}"
PROJECT_TAG = {"project": "undercover"}

session = boto3.Session()
iam = session.client("iam")
lambda_client = session.client("lambda")
logs = session.client("logs")
tagging = session.client("resourcegroupstaggingapi")

identity = session.client("sts").get_caller_identity()
print(f"Deploying as {identity['Arn']} in {session.region_name}")

A look at the budget before anything is deployed. The Budgets API is free to call.

In [ ]:
budgets = session.client("budgets").describe_budgets(AccountId=identity["Account"]).get("Budgets", [])

for budget in budgets:
    limit = float(budget["BudgetLimit"]["Amount"])
    actual = float(budget["CalculatedSpend"]["ActualSpend"]["Amount"])
    print(f"{budget['BudgetName']}: ${actual:,.2f} spent of ${limit:,.2f}")

if not budgets:
    print("No budget found. Create one in Billing and Cost Management, Budgets, before deploying.")

The app's claims routes need an `X-API-Key` header. The key is kept in `.env`, which git ignores, and this cell makes one the first time it runs. The same key goes into the function's environment at deploy time, and later into the AgentCore Gateway so that the agent's tool calls carry it.

In [ ]:
load_dotenv()

if not os.getenv("UNDERCOVER_API_KEY"):
    with open(".env", "a") as env_file:
        env_file.write(f"\nUNDERCOVER_API_KEY={secrets.token_urlsafe(32)}\n")
    load_dotenv()
    print("Made a new key and saved it in .env")

API_KEY = os.environ["UNDERCOVER_API_KEY"]
print(f"The API key is loaded, {len(API_KEY)} characters")

## 2. Build the package

Lambda runs Linux, so the dependencies have to be the Linux builds even when this machine is a Mac or a PC. `uv` can fetch wheels for another platform, which means no Docker. The package is a zip with the dependencies and `server.py` at its root.

In [ ]:
package_dir = Path("lambda_package")
zip_path = Path("undercover-app.zip")

shutil.rmtree(package_dir, ignore_errors=True)
zip_path.unlink(missing_ok=True)

!uv pip install --quiet --target {package_dir} --python-platform x86_64-manylinux2014 --python-version 3.12 --only-binary :all: fastapi mangum

shutil.copy("server.py", package_dir)
shutil.make_archive(zip_path.stem, "zip", package_dir)

print(f"{zip_path}: {zip_path.stat().st_size / 1_000_000:.1f} MB (Lambda allows 50 MB)")

## 3. Deploy

Four steps: the log group, the role the function runs as, the function itself, and its public URL. Each step can be run again safely, so a rerun updates the function in place.

### The log group

Lambda would make this log group by itself a few seconds after the first request, with logs kept forever. Making it here gives it a retention period, and it exists from a known moment, so the teardown can count on finding it.

In [ ]:
try:
    logs.create_log_group(logGroupName=LOG_GROUP)
    print(f"Created log group {LOG_GROUP}")
except logs.exceptions.ResourceAlreadyExistsException:
    print(f"Log group {LOG_GROUP} already exists")

logs.put_retention_policy(logGroupName=LOG_GROUP, retentionInDays=7)
print("Logs are kept for 7 days")

### The role

The trust policy lets the Lambda service assume the role. The permission policy allows one thing, writing to the function's own log group. The role cannot create log groups, so once the teardown has deleted the group, a late log write from Lambda cannot bring it back.

In [ ]:
trust_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {"Service": "lambda.amazonaws.com"},
            "Action": "sts:AssumeRole",
        }
    ],
}

try:
    role = iam.create_role(
        RoleName=ROLE_NAME,
        AssumeRolePolicyDocument=json.dumps(trust_policy),
        Description="The role the UnderCover app runs as on Lambda",
        Tags=[{"Key": key, "Value": value} for key, value in PROJECT_TAG.items()],
    )["Role"]
    print(f"Created role {ROLE_NAME}")
except iam.exceptions.EntityAlreadyExistsException:
    role = iam.get_role(RoleName=ROLE_NAME)["Role"]
    print(f"Role {ROLE_NAME} already exists")

logging_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": ["logs:CreateLogStream", "logs:PutLogEvents"],
            "Resource": f"arn:aws:logs:{session.region_name}:{identity['Account']}:log-group:{LOG_GROUP}:*",
        }
    ],
}

iam.put_role_policy(
    RoleName=ROLE_NAME,
    PolicyName="undercover-app-logging",
    PolicyDocument=json.dumps(logging_policy),
)
print("The role can write to the log group")

### The function

The handler is `server.handler`, the Mangum object at the bottom of `server.py`. The API key goes in as an environment variable. A brand new role takes a few seconds to reach Lambda, so the create call retries while Lambda reports that it cannot assume the role yet. When the function already exists, the code and the environment are two separate updates, and Lambda has to finish the first before it accepts the second.

In [ ]:
environment = {"Variables": {"UNDERCOVER_API_KEY": API_KEY}}


def create_function(zip_bytes):
    for attempt in range(10):
        try:
            return lambda_client.create_function(
                FunctionName=FUNCTION_NAME,
                Runtime="python3.12",
                Handler="server.handler",
                Role=role["Arn"],
                Code={"ZipFile": zip_bytes},
                Timeout=30,
                MemorySize=512,
                Environment=environment,
                Tags=PROJECT_TAG,
            )
        except lambda_client.exceptions.InvalidParameterValueException:
            print("Waiting for the new role to reach Lambda")
            time.sleep(3)
    raise RuntimeError("Lambda could not assume the role")


zip_bytes = zip_path.read_bytes()

try:
    lambda_client.get_function(FunctionName=FUNCTION_NAME)
    lambda_client.update_function_code(FunctionName=FUNCTION_NAME, ZipFile=zip_bytes)
    lambda_client.get_waiter("function_updated_v2").wait(FunctionName=FUNCTION_NAME)
    lambda_client.update_function_configuration(FunctionName=FUNCTION_NAME, Environment=environment)
    lambda_client.get_waiter("function_updated_v2").wait(FunctionName=FUNCTION_NAME)
    print(f"Updated function {FUNCTION_NAME}")
except lambda_client.exceptions.ResourceNotFoundException:
    create_function(zip_bytes)
    lambda_client.get_waiter("function_active_v2").wait(FunctionName=FUNCTION_NAME)
    print(f"Created function {FUNCTION_NAME}")

### The Function URL

A Function URL is a public HTTPS address that comes with the function, with no API Gateway and no extra charge. A public URL made through the API needs two permission statements on the function, one to call the URL and one to invoke the function through it. Without both, every request gets a 403.

In [ ]:
try:
    url_config = lambda_client.create_function_url_config(FunctionName=FUNCTION_NAME, AuthType="NONE")
except lambda_client.exceptions.ResourceConflictException:
    url_config = lambda_client.get_function_url_config(FunctionName=FUNCTION_NAME)

public_access = [
    {"StatementId": "public-url", "Action": "lambda:InvokeFunctionUrl", "FunctionUrlAuthType": "NONE"},
    {"StatementId": "public-invoke", "Action": "lambda:InvokeFunction", "InvokedViaFunctionUrl": True},
]

for statement in public_access:
    try:
        lambda_client.add_permission(FunctionName=FUNCTION_NAME, Principal="*", **statement)
    except lambda_client.exceptions.ResourceConflictException:
        pass

url = url_config["FunctionUrl"]
print(url)

## 4. Test it

The first request is a cold start: Lambda unpacks the zip and imports FastAPI before it answers. The requests after it reuse the warm function. The claims routes answer 401 without the key.

In [ ]:
def get(path, key=None):
    headers = {"X-API-Key": key} if key else {}
    start = time.perf_counter()
    response = requests.get(url + path, headers=headers, timeout=30)
    elapsed = time.perf_counter() - start
    print(f"GET /{path:<18} {'with key' if key else 'no key':<10} {response.status_code}   {elapsed:.2f}s")
    return response


get("health")
get("health")
no_key = get("claims")
claims = get("claims", API_KEY)
claim = get("claims/CLM-1002", API_KEY)
missing = get("claims/CLM-9999", API_KEY)
home = get("")

In [ ]:
print(json.dumps(claim.json(), indent=2))
print()
print(f"All claims: {[c['claim_id'] for c in claims.json()]}")
print(f"Without the key: {no_key.json()}")
print(f"Missing claim: {missing.json()}")
print(f"Home page: {home.headers['content-type']}, {len(home.text)} characters")

FastAPI publishes an OpenAPI document for the app. This is what the AgentCore Gateway will read later to turn these routes into tools for the agent.

In [ ]:
openapi = requests.get(url + "openapi.json", timeout=30).json()

for path, operations in openapi["paths"].items():
    for method, operation in operations.items():
        print(f"{method.upper():<5}{path:<28}{operation['operationId']:<16}{operation['description']}")

Open the URL in a browser too. `/docs` has FastAPI's interactive API page, served from Lambda.

## 5. What now exists in the account

Everything is found by name, with the `undercover` prefix, and then once more by the `project=undercover` tag across every service. Nothing here relies on remembering what was created.

In [ ]:
def find_functions(prefix=PREFIX):
    pages = lambda_client.get_paginator("list_functions").paginate()
    return [f["FunctionName"] for page in pages for f in page["Functions"] if f["FunctionName"].startswith(prefix)]


def find_roles(prefix=PREFIX):
    pages = iam.get_paginator("list_roles").paginate()
    return [r["RoleName"] for page in pages for r in page["Roles"] if r["RoleName"].startswith(prefix)]


def find_log_groups(prefix=PREFIX):
    pages = logs.get_paginator("describe_log_groups").paginate(logGroupNamePrefix=f"/aws/lambda/{prefix}")
    return [g["logGroupName"] for page in pages for g in page["logGroups"]]


def find_tagged():
    tag_filter = [{"Key": key, "Values": [value]} for key, value in PROJECT_TAG.items()]
    pages = tagging.get_paginator("get_resources").paginate(TagFilters=tag_filter)
    return [r["ResourceARN"] for page in pages for r in page["ResourceTagMappingList"]]


def show_resources():
    found = {
        "Lambda functions": find_functions(),
        "IAM roles": find_roles(),
        "Log groups": find_log_groups(),
        "Tagged project=undercover": find_tagged(),
    }
    for label, names in found.items():
        print(f"{label + ':':<28}{', '.join(names) or 'none'}")


show_resources()

The function's own report lines from the log group, one per request, with the time billed and the memory used. A cold start shows up as `Init Duration`. Log lines take a few seconds to arrive, and the cell shows the latest eight.

In [ ]:
pages = logs.get_paginator("filter_log_events").paginate(logGroupName=LOG_GROUP, filterPattern="REPORT")
events = [event for page in pages for event in page["events"]][-8:]

for event in events:
    print(event["message"].strip().replace("\t", "  "))

if not events:
    print("No log lines yet. Run this cell again in a few seconds.")

## 6. Teardown (optional)

Run this when you are done. It deletes what this notebook made: the function, the role and the log group whose names start with `undercover-app`, found with the same searches as above. The other labs share the `undercover` prefix and have a teardown of their own, so this one leaves them alone. Deleting a function removes its Function URL and its permission statements with it.

First, what will be deleted.

In [ ]:
to_delete = {
    "Lambda functions": find_functions(FUNCTION_NAME),
    "IAM roles": find_roles(FUNCTION_NAME),
    "Log groups": find_log_groups(FUNCTION_NAME),
}

for label, names in to_delete.items():
    print(f"{label + ':':<28}{', '.join(names) or 'none'}")

Then the delete.

In [ ]:
for name in to_delete["Lambda functions"]:
    lambda_client.delete_function(FunctionName=name)
    print(f"Deleted function {name}")

for name in to_delete["IAM roles"]:
    for policy in iam.list_attached_role_policies(RoleName=name)["AttachedPolicies"]:
        iam.detach_role_policy(RoleName=name, PolicyArn=policy["PolicyArn"])
    for policy_name in iam.list_role_policies(RoleName=name)["PolicyNames"]:
        iam.delete_role_policy(RoleName=name, PolicyName=policy_name)
    iam.delete_role(RoleName=name)
    print(f"Deleted role {name}")

for name in to_delete["Log groups"]:
    logs.delete_log_group(logGroupName=name)
    print(f"Deleted log group {name}")

The same searches again, and a request to the URL, which no longer answers. The tag search reads an index that can take a few seconds to notice a delete, so the cell waits for it to catch up.

In [ ]:
deadline = time.time() + 60
while any(f"function:{FUNCTION_NAME}" in arn for arn in find_tagged()) and time.time() < deadline:
    time.sleep(3)

show_resources()

response = requests.get(url + "health", timeout=30)
print(f"\nGET /health now returns {response.status_code}")